In [1]:
%pip -q install "transformers==5.20.2" "trl==1.16.0" "peft==0.21.2" accelerate bitsandbytes datasets sentencepiece

Imports

In [2]:
import torch
import transformers
import trl
import peft
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Select a T4 GPU under Runtime → Change runtime type, then reconnect.")
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Transformers: 5.20.2
TRL: 1.16.0
PEFT: 0.21.2
CUDA available: True
GPU: Tesla T4


In [3]:
# Original model and dataset
model_name = "NousResearch/Llama-2-7b-chat-hf"
dataset_name = "mlabonne/guanaco-llama2-1k"
new_model = "Llama-2-7b-chat-finetune"

# LoRA settings (original values)
lora_r = 64
lora_alpha = 16
lora_dropout = 0.1

# QLoRA: train small adapters over an NF4 4-bit quantized base model
use_4bit = True
bnb_4bit_compute_dtype = "float16"  # T4 supports FP16, not native BF16
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

# Training configuration (adjusted for ~16 GB Colab T4)
output_dir = "./results"
num_train_epochs = 1
fp16 = True
bf16 = False
per_device_train_batch_size = 1      # Reduced from 4 to reduce out-of-memory risk
gradient_accumulation_steps = 4      # Effective batch size remains 4
learning_rate = 2e-4
weight_decay = 0.001
optim = "paged_adamw_32bit"
lr_scheduler_type = "cosine"
max_steps = -1
warmup_ratio = 0.03                # Now passed to SFTConfig as warmup_steps (float fraction)
gradient_checkpointing = True
max_grad_norm = 0.3
group_by_length = True
logging_steps = 25
save_steps = 100                   # Positive number; checkpoint every 100 optimizer steps

# Modern TRL uses SFTConfig(max_length=...) instead of SFTTrainer(max_seq_length=...)
max_seq_length = 512               # Bounded to avoid T4 memory spikes
packing = False

device_map = {"": 0}

In [4]:
# Dataset: the original 'text' column is used for supervised fine-tuning
dataset = load_dataset(dataset_name, split="train")
if "text" not in dataset.column_names:
    raise ValueError(f"Expected a 'text' column, got {dataset.column_names}")
print("Training examples:", len(dataset))

# QLoRA 4-bit quantization
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)
major, minor = torch.cuda.get_device_capability(0)
print(f"GPU compute capability: {major}.{minor}")
if major >= 8:
    print("This GPU can use bf16; current settings deliberately use fp16.")

# Load 7B base model quantized onto GPU 0
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map,
)
model.config.use_cache = False  # Required for gradient checkpointing

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)

# SFTConfig combines the old TrainingArguments and the SFT-specific settings.
training_arguments = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_strategy="steps",
    save_steps=save_steps,
    save_total_limit=2,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_steps=warmup_ratio,       # 0.03 = first 3% of training steps
    train_sampling_strategy="group_by_length" if group_by_length else "random",
    lr_scheduler_type=lr_scheduler_type,
    gradient_checkpointing=gradient_checkpointing,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="text",
    max_length=max_seq_length,
    packing=packing,
    report_to="none",               # Avoid requiring TensorBoard/W&B
)

# Updated TRL API: use processing_class rather than tokenizer.
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments,
)

trainer.model.print_trainable_parameters()
trainer.train()

# Save only the LoRA adapter + the tokenizer (the base model isn't copied here).
trainer.save_model(new_model)
tokenizer.save_pretrained(new_model)
print("Saved LoRA adapter and tokenizer to:", new_model)

README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…): reconstructing file:   0%|          |  0.00B /  967kB            

data/train-00000-of-00001-9ad84bb9cf65a4(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Training examples: 1000
GPU compute capability: 7.5


...

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

trainable params: 33,554,432 || all params: 6,771,970,048 || trainable%: 0.4955


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [5]:
import torch
from collections import Counter

# Check existing trainable parameter types
print("Before:", Counter(
    str(p.dtype)
    for p in trainer.model.parameters()
    if p.requires_grad
))

# Keep LoRA trainable parameters in FP32
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
        p.grad = None

print("After:", Counter(
    str(p.dtype)
    for p in trainer.model.parameters()
    if p.requires_grad
))

# Reset optimizer state from the failed attempt
trainer.optimizer = None
trainer.lr_scheduler = None

# Retry training
trainer.train()

Before: Counter({'torch.bfloat16': 128})
After: Counter({'torch.float32': 128})


Step,Training Loss
25,1.522056
50,1.606957
75,1.247126
100,1.492252
125,1.210044
150,1.381469
175,1.181042
200,1.531545
225,1.177600
250,1.454258


TrainOutput(global_step=250, training_loss=1.38043505859375, metrics={'train_runtime': 1270.9835, 'train_samples_per_second': 0.787, 'train_steps_per_second': 0.197, 'total_flos': 1.3875997089742848e+16, 'train_loss': 1.38043505859375, 'epoch': 1.0})

In [6]:
trainer.save_model(new_model)
tokenizer.save_pretrained(new_model)

('Llama-2-7b-chat-finetune/tokenizer_config.json',
 'Llama-2-7b-chat-finetune/tokenizer.json')

In [17]:

logging.set_verbosity(logging.CRITICAL)
from transformers import pipeline
prompt = "What is a large language model?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

<s>[INST] What is a large language model? [/INST]A large language model is a type of artificial intelligence model that is trained on a large dataset of text to generate language outputs that are coherent and natural-sounding. The model is trained to predict the next word in a sequence of text given the context of the previous words. This is done by using a neural network architecture that is trained on a large dataset of text, such as a corpus of books or a collection of articles. The model is then fine-tuned on a smaller dataset of text to improve its performance on a specific task, such as text generation or language translation. 


In [14]:
print(trainer.state.global_step)
print(trainer.state.log_history[-5:])

250
[{'loss': 1.1810421752929687, 'grad_norm': 0.16936366260051727, 'learning_rate': 4.484840700157295e-05, 'entropy': 1.1815854540467263, 'mean_token_accuracy': 0.7048237946629524, 'num_tokens': 260360.0, 'epoch': 0.7, 'step': 175}, {'loss': 1.5315452575683595, 'grad_norm': 0.7361836433410645, 'learning_rate': 2.112793368281799e-05, 'entropy': 1.4671312218904495, 'mean_token_accuracy': 0.650717031955719, 'num_tokens': 279496.0, 'epoch': 0.8, 'step': 200}, {'loss': 1.1775999450683594, 'grad_norm': 0.10828847438097, 'learning_rate': 5.6423333488018095e-06, 'entropy': 1.1840725874900817, 'mean_token_accuracy': 0.7056987357139587, 'num_tokens': 327073.0, 'epoch': 0.9, 'step': 225}, {'loss': 1.454258270263672, 'grad_norm': 0.5961107015609741, 'learning_rate': 8.426222418311814e-09, 'entropy': 1.4682094687223435, 'mean_token_accuracy': 0.6505933529138566, 'num_tokens': 348246.0, 'epoch': 1.0, 'step': 250}, {'train_runtime': 1270.9835, 'train_samples_per_second': 0.787, 'train_steps_per_seco

In [15]:
import torch

finetuned_model = trainer.model
finetuned_model.eval()

prompt = "<s>[INST] What is a large language model? [/INST]"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(next(finetuned_model.parameters()).device)

def generate_answer():
    with torch.no_grad():
        output = finetuned_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

# Test without fine-tuning
print("=== BASE MODEL ===")

with finetuned_model.disable_adapter():
    print(generate_answer())

# Test with fine-tuning
print("\n=== FINE-TUNED MODEL ===")
print(generate_answer())

=== BASE MODEL ===
 A large language model (LLM) is a type of artificial intelligence (AI) model that is trained on a large dataset of text to generate language outputs that are coherent and natural-sounding. everybody has heard of chatbots, but the models behind them are often opaque and difficult to understand. Large language models are designed to overcome these limitations by using machine learning algorithms

=== FINE-TUNED MODEL ===
A large language model (LLM) is a type of artificial intelligence (AI) that has been trained on a vast amount of text data to generate human-like responses. It can be used for various applications such as chatbots, language translation, and content generation. The term "large" refers to the size of the model's neural network architecture, which typically consists of multiple layers of


In [1]:
del model
del pipe
del trainer
import gc
gc.collect()
gc.collect*()

NameError: name 'model' is not defined